# Push to Drive

Uploads local `src/` notebooks from this repo to a configured Google Drive folder.

**Run order:** authenticate → configure → push

In [ ]:
!pip install -q google-auth google-auth-oauthlib google-api-python-client gitpython

In [ ]:
from google.colab import userdata

# === CONFIGURE (values come from Colab Secrets — left panel 🔑) ===
DRIVE_FOLDER_ID  = userdata.get('DRIVE_FOLDER_ID')   # Drive destination folder
GITHUB_TOKEN     = userdata.get('GITHUB_TOKEN')       # repo-scoped PAT
GITHUB_USER      = userdata.get('GITHUB_USER') or 'albertolicea00'
GITHUB_REPO      = userdata.get('GITHUB_REPO') or 'agentscripts'
LOCAL_REPO_PATH  = '/content/agentscripts'
GITHUB_REPO_URL  = f'https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git'


In [ ]:
from google.colab import auth, drive
auth.authenticate_user()
drive.mount("/content/drive")

In [ ]:
import os, subprocess
import google.auth
from googleapiclient.discovery import build

creds, _ = google.auth.default()
service = build("drive", "v3", credentials=creds)

# Clone repo
if not os.path.exists(LOCAL_REPO_PATH):
    subprocess.run(["git", "clone", GITHUB_REPO_URL, LOCAL_REPO_PATH], check=True)

# Collect all notebooks under src/
notebooks = []
src_path = os.path.join(LOCAL_REPO_PATH, "src")
for root, _, files in os.walk(src_path):
    for f in files:
        if f.endswith(".ipynb"):
            notebooks.append(os.path.join(root, f))

print(f"Found {len(notebooks)} notebooks to push:")
for nb in notebooks:
    print(" ", nb.replace(LOCAL_REPO_PATH + "/", ""))

In [ ]:
from googleapiclient.http import MediaFileUpload

# Check existing files in Drive to avoid duplicates (update instead of create)
existing = {}
results = service.files().list(
    q=f"'{DRIVE_FOLDER_ID}' in parents and trashed=false",
    fields="files(id, name)",
).execute()
for f in results.get("files", []):
    existing[f["name"]] = f["id"]

for nb_path in notebooks:
    name = os.path.basename(nb_path)
    media = MediaFileUpload(nb_path, mimetype="application/x-ipynb+json")

    if name in existing:
        service.files().update(fileId=existing[name], media_body=media).execute()
        print(f"  Updated: {name}")
    else:
        meta = {"name": name, "parents": [DRIVE_FOLDER_ID]}
        service.files().create(body=meta, media_body=media).execute()
        print(f"  Created: {name}")

print("Done.")